In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import json
import requests
import geojsonio
import time
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
import json
import sys
import errno

load_dotenv()

In [ ]:
BASE_PATH = Path('/projectnb/planet/PLSP')
TEST_BASE_PATH = Path('/projectnb/modislc/users/fache/planet/')

In [ ]:
base_url = "https://api.planet.com/data/v1"
orders_url = 'https://api.planet.com/compute/ops/orders/v2'
stats_url = "{}/stats".format(base_url)
quick_url = "{}/quick-search".format(base_url)

# Helpers

In [ ]:
def p(data):
    print(json.dumps(data, indent=2))

In [ ]:
def create_geojson_file(row):
    return BASE_PATH / 'geojson' / f'{row["site"]}.geojson'

def create_raw_path(row):
    return BASE_PATH / 'raw' / row["site"]

# Load Site Metadata

In [ ]:
metadata_df = pd.DataFrame()
metadata_df['site'] = ['Walnut_Gulch_Kendall_Grasslands', 'Willard_Juniper_Savannah', 'Mountainair_Pinyon-Juniper_Woodland', 'Santa_Rita_Grassland', 'Santa_Rita_Mesquite', 'Sevilleta_shrubland', 'Walnut_Gulch_Lucky_Hills_Shrub', 'ARM_Southern_Great_Plains_site-_Lamont']
metadata_df['geojson_file'] = metadata_df.apply(create_geojson_file, axis=1)
metadata_df['raw_path'] = metadata_df.apply(create_raw_path, axis=1)
# metadata_df['geometry'] = metadata_df.apply(load_geometry_from_geojson, axis=1)
# metadata_gdf = gpd.GeoDataFrame(metadata_df, geometry='geometry')

In [ ]:
metadata_df.head(10)

# Load Planet Scene Counts

In [ ]:
def setup_filter(coords, minyear, maxyear):
    
    geometry_filter = {
        "type": "GeometryFilter",
        "field_name": "geometry",
        "config": {
          "type": "Polygon",
          "coordinates": coords
        }
    }
    
    date_filter = {
        "type": "DateRangeFilter", # Type of filter -> Date Range
        "field_name": "acquired", # The field to filter on: "acquired" -> Date on which the "image was taken"
        "config": {
            "gte": "{}-01-01T00:00:00.000Z".format(minyear), # "gte" -> Greater than or equal to
            "lt":"{}-01-01T00:00:00Z".format(maxyear)
            }
        }
    
    ground_control =  {
        "type": "StringInFilter",
        "config": ["true"],
        "field_name": "ground_control" # NOTE
    }
    
    quality_category = {
        "type": "StringInFilter",
        "config": ["standard"],
        "field_name": "quality_category" # NOTE
    }
    
    cloud_cover =  {
        "type": "RangeFilter",
        "field_name": "cloud_cover",
        "config": {"gte": 0, "lte": 0.5} # NOTE
    }
    
    asset = {
        "type": "AssetFilter",
        "config": [
            "ortho_analytic_4b_sr", # NOTE before analytic_sr 
            "ortho_analytic_4b", # NOTE before analytic
            "ortho_udm2" # NOTE udm2 no longer exists, for instance, is only available globally through July 2018."
            # 
        ]
    }

    permission = {
        "type":"PermissionFilter",
        "config":[
            "assets:download" # NOTE
        ]
    }

    and_filter = {
        "type": "AndFilter",
        "config": [ 
            cloud_cover,
            quality_category,
            ground_control,
            date_filter,
            geometry_filter,
            permission,
            asset
        ]
    }
    
    return and_filter

In [ ]:
def read_geometry(path):
    
    if(False == os.path.exists(path)):
        sys.exit("GeoJSON path doesn't exist: {}".format(path))
        
    with open(path, "r") as file1:
        geo = json.load(file1)
        
    return geo

In [ ]:
key = os.environ.get("PLANET_API_KEY")
print("key exists:", key is not None)
print("key length:", len(key) if key else 0)
print("key repr:", repr(key))

PLANET_API_KEY = os.getenv('PLANET_API_KEY')

# Setup the session
session = requests.Session()
# Authenticate
session.auth = (PLANET_API_KEY, "")

print("GET:", session.get("https://api.planet.com/data/v1").status_code)

# Make a GET request to the Planet Data API
res = session.get(base_url)
# Response status code
if(res.status_code != 200):
    print("Cannot cannot to base server {} with status code {}".format(base_url, res.status_code))
    sys.exit("Cannot cannot to base server {} with status code {}".format(base_url, res.status_code))
else:
    print("Base server is alive.")

p(res.json())

In [ ]:
output_dir = TEST_BASE_PATH / 'rawImage'

min_year = 2025
max_year = 2026

In [ ]:
for i, row in metadata_df.iterrows():
    start_time = time.time()

    print(f'\n\n{"-"*10}{row["site"]}{"-"*10}')
    

    output_site_dir = os.path.join(output_dir, row['site'])
    if not os.path.exists(output_site_dir):
        try:
            os.makedirs(output_site_dir)
        except OSError as exc: # Guard against race condition
            if exc.errno != errno.EEXIST:
                raise
    
    print("site output_site_dir created: {}".format(output_site_dir))


    geo = read_geometry(row['geojson_file'])
    
    for x in geo['features']: # get all geometry features, for each one
        feature_name = x['properties']['f']
        feature_coords = x['geometry']['coordinates']
        
        filter = setup_filter(feature_coords, min_year, max_year)
        
        # print("Filter config for search:")
        # p(filter)

        # ---------- get some quick stats
        
        print("---- stat count ----")
        request = {
            "interval" : "year",
            "filter" : filter,
            "item_types" : ["PSScene"] # NOTE replaces PSScene4Band https://community.planet.com/product-updates/event-psscene-migration-workshop-161
        }

        # Send the POST request to the API stats endpoint
        res = session.post(stats_url, json=request)

        # print(res.status_code)
        # print(res.text)
        # print(res.request.headers)
        # print(res.request.body)
        
        if(res.status_code != 200):
            sys.exit("Stats search failed with code {}".format(res.status_code))

        for bucket in res.json()['buckets']:
            print("start_time: {} count: {}".format(bucket["start_time"], bucket["count"]))
        
        # ---------- perform real asset search

        print("---- quick search ----")
        request = {
            "filter" : filter,
            "item_types" : ["PSScene"]
        }

        # Send the POST request to the API quick search endpoint
        res = session.post(quick_url, json=request)
        
        if(res.status_code != 200):
            sys.exit("Quick search failed with code {}".format(res.status_code))
            
        quick_search_results_json = res.json()
        
        filename = "{}_quick_search_result_{}_{}.json".format(feature_name.replace(" ", "_"), min_year, max_year)
        with open(os.path.join(output_site_dir, filename), 'w') as outfile:
            json.dump(quick_search_results_json, outfile)
            print('quick-search output file created: {}'.format(filename))
        
        # ---------- get all assets that need to be downloaded

        print('---- assembling feature ids to download ----')
        features = quick_search_results_json['features']

        if(len(features) == 0):
            sys.exit("0 IDs returned in quick search.")

        id_list = []
        num_next_urls = 0
        while(len(quick_search_results_json["features"]) > 0):
            print('iteration: {}'.format(num_next_urls))
            
            for x in quick_search_results_json["features"]: # go through all features and collect all scene ids
                id_list.append(x["id"])
            
            # Assign the "_links" -> "_next" property (link to next page of results) to a variable 
            next_url = quick_search_results_json["_links"]["_next"]
            if (next_url is None):
                break
            num_next_urls += 1
            
            # from the next url, if there are results, update quick_search_results_json and append to output_site_dir
            time.sleep(5)
            res = session.get(next_url)
            
            if(res.status_code != 200):
                sys.exit("Next page retrieval failed with code {}".format(res.status_code))
                
            quick_search_results_json = res.json()
            with open(os.path.join(output_site_dir, filename), 'a') as outfile:
                json.dump(quick_search_results_json, outfile)
            
            # output_site_dir is now on the next page, keep looping for more features

        print("Total IDs: {}".format(len(id_list)))
        print('Number of _next urls triggered: {}'.format(num_next_urls))

        print('chunking ids')
        


    print("---- %s seconds ----" % (time.time() - start_time))

    if i == 0:
        break

In [ ]:
len(id_list)